# exp007 — Phase 1, step 1b: why is training data loading limited to ~45 examples/s? (Kaggle T4 x2, 4 CPUs)

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, notebook **private** (`kaggle kernels push`). Config: `configs/exp007_loader_diag.yaml`.
Heavy files live in `/tmp/ks`; only small result files go to `/kaggle/working/outputs/exp007_loader_diag/`.
Downloads: ~0.4 GB of sparse volume chunks (S3) + ~1 MB of labels (HF). Expected duration 30-45 min (rerun: the first run lost its results to a notebook-exit-code bug, see the commit message).
Throwaway models are trained for 150 steps only to measure speed. Numbers only, no images.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
echo "cpus: $(nproc)"; free -g | sed -n 2p; df -h /kaggle/working | tail -1; python --version
mkdir -p /tmp/ks
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /tmp/ks/harness
git -C /tmp/ks/harness log -1 --format='harness %H'

In [ ]:
import os, yaml, pathlib
SCR = "/tmp/ks"
H = pathlib.Path(SCR + "/harness")
cfg7 = yaml.safe_load((H / "configs/exp007_loader_diag.yaml").read_text())
cfg6 = yaml.safe_load((H / cfg7["base_config"]).read_text())
os.environ.update({
    "H": str(H), "EXP7_CFG": str(H / "configs/exp007_loader_diag.yaml"), "EXP6_CFG": str(H / cfg7["base_config"]),
    "VILLA_SHA": cfg6["upstream"]["villa_commit"], "PY_VER": cfg6["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg6["upstream"]["uv_sync_args"]),
    "VES": SCR + "/villa/vesuvius", "VILLA": SCR + "/villa",
    "DATA": SCR + "/train_data", "DATA_UNC": SCR + "/train_data_unc", "OUT": SCR + "/exp007_loader_diag",
    "RESULTS": "/kaggle/working/outputs/exp007_loader_diag",
    "TIMEOUT": str(cfg7["timeout_seconds"]), "CPU_INTERVAL": str(cfg7["cpu_monitor_interval_s"]),
})
print({k: os.environ[k] for k in ["VILLA_SHA", "TIMEOUT", "CPU_INTERVAL"]})

## 1. Upstream code (villa, pinned commit) + environment

In [ ]:
%%bash
set -e
git init -q /tmp/ks/villa && cd /tmp/ks/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -5
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), 'torch threads', torch.get_num_threads())"

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing here runs `vesuvius.accept_terms`.

## 3. Sparse training data (same as exp006), then the uncompressed variant

In [ ]:
%%bash
set -eo pipefail
cd "$VES" && uv run --no-sync python "$H/scripts/exp006_prepare_train_data.py" --config "$EXP6_CFG" --out "$DATA" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -40
du -sh "$DATA"

In [ ]:
%%bash
set -eo pipefail
cd "$VES" && uv run --no-sync python "$H/scripts/exp007_make_variants.py" --src "$DATA" --dst "$DATA_UNC" 2>&1 | tail -30
du -sh "$DATA" "$DATA_UNC"

## 4. Training configs (one per variant) validated with villa's TrainingConfig

In [ ]:
%%bash
set -eo pipefail
mkdir -p "$OUT/runs" "$OUT/logs"
cd "$VES" && uv run --no-sync python "$H/scripts/exp007_make_configs.py" --config "$EXP7_CFG" --villa "$VILLA" --data "$DATA" --data-unc "$DATA_UNC" --out "$OUT" | tail -60

## 5. Loader-only profile (cProfile with/without augmentation, DataLoader at 4 workers, augmentation-off bound). A failing section is recorded, not fatal.

In [ ]:
%%bash
cd "$VES"
mkdir -p "$RESULTS/logs"
uv run --no-sync python "$H/scripts/exp007_loader_profile.py" --train-config "$OUT/base_w4.json" --params "$EXP7_CFG" --out "$OUT/loader_profile_default.json" > "$OUT/logs/loader_profile_default.log" 2>&1 < /dev/null
echo "loader profile: exit $?"; grep -E "^\[sweep\]|FAILED" "$OUT/logs/loader_profile_default.log" | cut -c1-200
cp "$OUT/loader_profile_default.json" "$RESULTS/" 2>/dev/null || true
python3 -c "import json; print('LOADER_PROFILE_JSON', json.dumps(json.load(open('$OUT/loader_profile_default.json'))))" 2>&1 | cut -c1-6000
true

## 6. End-to-end trainer benchmarks (150 steps, 30 warm-up excluded) with a CPU monitor around every run

In [ ]:
%%bash
cd "$VES"
python3 -c "
import json
for r in json.load(open('$OUT/runs.json')):
    envs = ','.join(f'{k}={v}' for k, v in r['env'].items()) or 'NONE'
    print(r['id'], r['gpus'], r['config'], envs)" > "$OUT/runs.txt"
while read -r id gpus cfgp envs; do
  echo "=== $id ($gpus GPU, env=$envs) $(date +%T) ==="
  python3 "$H/scripts/exp007_cpu_monitor.py" --out "$OUT/logs/cpu_$id.jsonl" --interval "$CPU_INTERVAL" < /dev/null &
  MON=$!
  if [ "$envs" = NONE ]; then EV=""; else EV="${envs//,/ }"; fi
  if [ "$gpus" = 1 ]; then
    env CUDA_VISIBLE_DEVICES=0 $EV timeout "$TIMEOUT" uv run --no-sync python -m vesuvius.ink_detection.training.train "$cfgp" > "$OUT/logs/$id.log" 2>&1 < /dev/null
  else
    env $EV timeout "$TIMEOUT" uv run --no-sync accelerate launch --num_processes "$gpus" --module vesuvius.ink_detection.training.train "$cfgp" > "$OUT/logs/$id.log" 2>&1 < /dev/null
  fi
  echo "exit status $? | benchmark json: $(ls "$OUT/runs/${id}_benchmark.json" 2>/dev/null || echo MISSING)"
  kill $MON 2>/dev/null || true; wait $MON 2>/dev/null || true
  echo "BENCH_JSON $id $(tr -d ' \n' < "$OUT/runs/${id}_benchmark.json" 2>/dev/null)"
done < "$OUT/runs.txt"
mkdir -p "$RESULTS/runs" "$RESULTS/logs"
cp "$OUT"/runs/*_benchmark.json "$RESULTS/runs/" 2>/dev/null || true
cp "$OUT"/logs/cpu_*.jsonl "$RESULTS/logs/" 2>/dev/null || true
cp "$OUT/runs.json" "$RESULTS/" 2>/dev/null || true
true

## 7. Summary (numbers only) and the small files kept as notebook output

In [ ]:
%%bash
cd "$VES" && uv run --no-sync python "$H/scripts/exp007_summarize.py" --out "$OUT"
cp "$DATA/train_data.json" "$OUT/train_data.json"
mkdir -p "$RESULTS/runs" "$RESULTS/logs"
cp "$OUT/summary.json" "$OUT/runs.json" "$OUT/train_data.json" "$OUT"/loader_profile_*.json "$RESULTS/" 2>/dev/null || true
cp "$OUT"/runs/*_benchmark.json "$RESULTS/runs/" 2>/dev/null || true
cp "$OUT"/logs/*.log "$OUT"/logs/cpu_*.jsonl "$RESULTS/logs/" 2>/dev/null || true
echo "=== files kept as notebook output ==="; (cd "$RESULTS" && find . -type f | sort | xargs sha256sum)
du -sh "$RESULTS"